# Model Trainging – Heart Disease Dataset
*Training and evaluation of models for Cardio - Risk Prediction project*  

---

## Table of Contents


<a id='imports'></a>
## Reproducibility & Imports  
---

In [ ]:
# Reproducibility
import os, sys
import pandas as pd
import joblib

# Imports
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))

# Models & tools
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

SEED = 42

In [2]:
df = pd.read_csv('../data/df_mix.csv')
df.head()

,PCA_1,tSNE_1,UMAP_1,DEATH_EVENT
0,-1.359239,-0.390639,7.591517,0
1,-1.521572,1.934435,8.070341,0
2,-1.451933,-0.382971,7.543156,0
3,-1.192678,0.936099,7.555950,0
4,-1.816253,0.799211,7.809438,0


Targets setup (for binary and mutliclass)

In [3]:
df_copy = df.copy()
TARGET_COL = 'DEATH_EVENT'

X = df_copy.drop(columns=[TARGET_COL])
y = df_copy[TARGET_COL]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

Data split

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Split shapes:")
print("  X_tr:", X_tr.shape, "| X_te:", X_te.shape)

Split shapes:
  X_tr: (201, 3) | X_te: (51, 3)


<a id='random-forest'></a>
---
## Random Forest

In [5]:
params_grid = {
    'n_estimators': [700, 900, 1000, 1200],
    'max_depth': [6, 7, 8, 9],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['sqrt', 0.4, 0.6, 0.8],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced'],
}

In [ ]:
grid = GridSearchCV(
    estimator = RandomForestClassifier(n_jobs=-1, random_state=SEED),
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
best = grid.best_estimator_

y_pred = best.predict(X_te)
score = round((accuracy_score(y_te, y_pred))*100, 3)

print('Best parameters:', grid.best_params_)
print('Accuracy:', score)

# save the best model
os.makedirs('../outputs/models', exist_ok=True)
model_path = '../outputs/models/rf.joblib'
joblib.dump(best, model_path)
print('Model saved to: ', model_path)

Fitting 5 folds for each of 2304 candidates, totalling 11520 fits
Best parameters: {'bootstrap': False, 'class_weight': None, 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 700}
Accuracy: 100.0
Model saved to:  ../outputs/models/unbalanced/rf.joblib


<a id='adaboost'></a>
## AdaBoost

In [15]:
params_grid = {
    'n_estimators': [50, 100, 200, 600],
    'learning_rate': [0.01, 0.05, 0.1],
    'estimator__max_depth': [1, 2, 3, 4],
    'estimator__min_samples_leaf': [1, 2, 3],
    'estimator__class_weight': [None, 'balanced'],
}

In [16]:
grid = GridSearchCV(
    estimator = AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=SEED), random_state=SEED),
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
best = grid.best_estimator_

y_pred = best.predict(X_te)
score = round((accuracy_score(y_te, y_pred))*100, 3)

print('Best params:', grid.best_params_)
print('Accuracy:', score)

# save the best model
os.makedirs('../outputs/models', exist_ok=True)
model_path = '../outputs/models/ada.joblib'
joblib.dump(best, model_path)
print('Model saved to: ', model_path)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best params: {'estimator__class_weight': None, 'estimator__max_depth': 3, 'estimator__min_samples_leaf': 1, 'learning_rate': 0.05, 'n_estimators': 200}
Accuracy: 98.039
Model saved to:  ../outputs/models/ada.joblib


<a id='xgboost'></a>
## XGBoost


In [11]:
params_grid = {
    'n_estimators': [40, 50, 100, 110, 150],   
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4, 5, 6],
    'min_child_weight': [1, 2, 3, 4, 5, 6],
    'subsample': [0.5, 1.0],
    'reg_lambda': [0.01, 0.05, 0.5, 0.1, 4.0, 5.0],
}


In [ ]:
xgb = XGBClassifier(
    tree_method = 'hist',
    objective = 'binary:logistic',
    eval_metric = 'logloss',
    n_jobs = -1,
    random_state = SEED,
)

grid = GridSearchCV(
    estimator = xgb,
    param_grid = params_grid,
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring = 'accuracy',
    n_jobs = -1,
    verbose = 10,
)

grid.fit(X_tr, y_tr)
best = grid.best_estimator_

y_pred = best.predict(X_te)
score = round((accuracy_score(y_te, y_pred))*100, 3)

print('Best params:', grid.best_params_)
print('Accuracy:', score)

# save the best model
os.makedirs('../outputs/models', exist_ok=True)
model_path = '../outputs/models/xgb.joblib'
joblib.dump(best, model_path)
print('Model saved to: ', model_path)

Fitting 5 folds for each of 5400 candidates, totalling 27000 fits
Best params: {'learning_rate': 0.1, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 100, 'reg_lambda': 0.01, 'subsample': 1.0}
Accuracy: 100.0
Model saved to:  ../outputs/models/unbalanced/xgb.joblib
